# Trading Code Generator (Simulated Environment)

This notebook implements **idea 3** from the week 4 exercise: a code generator that writes trading code to buy and sell equities in a simulated environment, based on a given API.

- **Input**: API specification (JSON) and optional strategy description.
- **Output**: Generated Python script that calls the API and implements the strategy.
- **Simulated environment**: A local mock API (`simulator.py`) so you can run the generated bot without real brokers.

In [2]:
# Imports
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API key loaded (starts with {openai_api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set. Set it in .env or environment to use code generation.")

# Use OpenAI; for OpenRouter use: OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openai_api_key) if openai_api_key else None
model = "gpt-4o"  # or gpt-4o-mini, etc.

OpenAI API key loaded (starts with sk-or-v1...)


## 1. Load API specification

The API spec defines endpoints (get_price, place_order, get_balance, get_positions), auth, and base URL. The simulator implements this spec locally.

In [4]:
# Load api_spec from week4 or current dir
spec_path = Path("api_spec.json") if Path("api_spec.json").exists() else Path("week4/api_spec.json")
with open(spec_path, "r") as f:
    api_spec = json.load(f)
print("Loaded API spec:", api_spec.get("name", ""))
print(json.dumps(api_spec, indent=2)[:800], "...")

Loaded API spec: ExampleEquitySim
{
  "name": "ExampleEquitySim",
  "base_url": "https://sim.example.com/api",
  "endpoints": {
    "get_price": {
      "path": "/market/price",
      "method": "GET",
      "params": [
        "symbol"
      ]
    },
    "place_order": {
      "path": "/orders",
      "method": "POST",
      "body": [
        "symbol",
        "side",
        "quantity",
        "order_type",
        "price_optional"
      ]
    },
    "cancel_order": {
      "path": "/orders/{order_id}/cancel",
      "method": "POST"
    },
    "get_balance": {
      "path": "/account/balance",
      "method": "GET"
    },
    "get_positions": {
      "path": "/account/positions",
      "method": "GET"
    }
  },
  "auth": {
    "type": "api_key_header",
    "header_name": "X-API-KEY",
    "api_key_placeholder": "<SIM_API ...


## 2. Build prompt and generate trading bot code

In [5]:
def build_prompt(api_spec: dict, strategy: str = "sma_crossover") -> str:
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    return f"""
You are a code-writing assistant. Produce a single, self-contained Python script that implements a trading bot for a *simulated* equities API. The API has this specification (JSON):

{json.dumps(api_spec, indent=2)}

Requirements:
1. Use only stdlib + `requests`. No other external dependencies.
2. Implement a simple strategy: {strategy}. For \"sma_crossover\": fetch prices over time (call get_price repeatedly if needed), compute short (e.g. 5) and long (e.g. 20) simple moving averages; when short SMA crosses above long SMA place a BUY (use a fraction of cash); when short crosses below long place a SELL to reduce position.
3. Build URLs as base_url + path. Use the auth header from the spec (e.g. X-API-KEY).
4. Include error handling and logging (print is fine).
5. Add a `--dry-run` flag that prints actions without placing orders.
6. Throttle requests and add configurable `min_time_between_trades_seconds`.
7. Add a short docstring at the top and a `if __name__ == "__main__":` block that runs a short demo (e.g. 30–60 seconds) in dry-run mode.
8. Output valid Python code only. No markdown or explanation around the code.

Timestamp: {now}
""".strip()


def extract_code(response: str) -> str:
    """Strip markdown code fences if present."""
    text = response.strip()
    for start in ("```python", "```"):
        if text.startswith(start):
            text = text[len(start):].lstrip()
        if text.endswith("```"):
            text = text[:-3].rstrip()
    return text


def sanity_check(code: str) -> bool:
    return (
        ("import requests" in code or "import urllib" in code)
        and "def " in code
        and ("place_order" in code or "order" in code.lower())
        and "if __name__" in code
    )

In [6]:
if not client:
    print("Skipping generation: no API client. Set OPENAI_API_KEY.")
else:
    strategy = "sma_crossover"
    user_prompt = build_prompt(api_spec, strategy)
    messages = [
        {"role": "system", "content": "You generate only runnable Python code. No markdown, no explanation."},
        {"role": "user", "content": user_prompt},
    ]
    print("Calling model...")
    response = client.chat.completions.create(model=model, messages=messages, max_tokens=8000)
    raw = response.choices[0].message.content or ""
    generated_code = extract_code(raw)
    if not sanity_check(generated_code):
        print("Warning: sanity check failed. Review the generated code.")
    out_path = Path("generated_trading_bot.py") if Path(".").resolve().name != "week4" else Path("generated_trading_bot.py")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(generated_code, encoding="utf-8")
    print(f"Saved to {out_path.absolute()}")
    print("--- Preview (first 1200 chars) ---")
    print(generated_code[:1200])
    print("...")

Calling model...
Saved to /Users/kelvini/Andela-LLM-Engineering/llm_engineering/week4/community-contributions/Wakanda-Kelvin-Isievwore/generated_trading_bot.py
--- Preview (first 1200 chars) ---
import requests
import time
import argparse
from collections import deque

API_KEY = '<SIM_API_KEY>'
BASE_URL = "https://sim.example.com/api"

HEADERS = {
    "X-API-KEY": API_KEY
}

SYMBOL = "AAPL"
SHORT_WINDOW = 5
LONG_WINDOW = 20
MIN_TIME_BETWEEN_TRADES_SECONDS = 60
CASH_ALLOCATION_FOR_TRADES = 0.1

def get_price(symbol):
    try:
        response = requests.get(f"{BASE_URL}/market/price", headers=HEADERS, params={"symbol": symbol})
        response.raise_for_status()
        return response.json().get('price')
    except requests.RequestException as e:
        print(f"Error getting price for {symbol}: {e}")
        return None

def place_order(symbol, side, quantity, order_type="market", price_optional=None):
    order_details = {
        "symbol": symbol,
        "side": side,
        "qua

## 3. Run the simulator and execute the generated bot

Start the local simulated API (in a background thread), then run the generated script so it talks to the simulator.

In [6]:
# Ensure week4 is on path when running from repo root
import sys
if not Path("simulator.py").exists() and Path("week4/simulator.py").exists():
    sys.path.insert(0, str(Path("week4").resolve()))
from simulator import start_simulator_background

SIM_PORT = 8765
SIM_API_KEY = "sim-key-123"
thread, base_url = start_simulator_background(port=SIM_PORT, api_key=SIM_API_KEY)
print(f"Simulator running at {base_url}")
print("Use base_url and X-API-KEY when running the generated bot.")

Simulator running at http://127.0.0.1:8765/api
Use base_url and X-API-KEY when running the generated bot.


In [7]:
# Run the generated bot: point it at the simulator and use dry-run or live run
import tempfile
import sys
bot_path = Path("generated_trading_bot.py") if Path("generated_trading_bot.py").exists() else Path("week4/generated_trading_bot.py")
if not bot_path.exists():
    print("No generated bot found. Run the generation cell first.")
else:
    env = os.environ.copy()
    env["SIM_API_KEY"] = SIM_API_KEY
    env["BASE_URL"] = base_url
    env["SIM_BASE_URL"] = base_url
    bot_src = bot_path.read_text(encoding="utf-8")
    # Patch hardcoded simulator URL and API key so the bot talks to our local simulator
    bot_src = bot_src.replace("https://sim.example.com/api", base_url)
    bot_src = bot_src.replace("<SIM_API_KEY>", SIM_API_KEY)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(bot_src)
        tmp_path = f.name
    try:
        result = subprocess.run(
            [sys.executable, tmp_path, "--dry-run"],
            env=env,
            capture_output=True,
            text=True,
            timeout=90,
            cwd=str(bot_path.parent) if str(bot_path.parent) != "." else ".",
        )
        print("STDOUT:", result.stdout or "(none)")
        if result.stderr:
            print("STDERR:", result.stderr)
        print("Return code:", result.returncode)
    finally:
        os.unlink(tmp_path)

STDOUT: Current price for AAPL: 149.26
Current price for AAPL: 150.0
Current price for AAPL: 149.35
Current price for AAPL: 146.42
Current price for AAPL: 146.58
Current price for AAPL: 145.17
Current price for AAPL: 144.8
Current price for AAPL: 146.71
Current price for AAPL: 148.56
Current price for AAPL: 147.46
Current price for AAPL: 147.62
Current price for AAPL: 146.85

Return code: 0
